# 00 · Setup and smoke test

Checks the GPU against the reference architecture table, records which Triton and PyTorch APIs
exist on this stack, compiles one matmul config to see where registers, spills, shared memory and
the PTX live on the compiled kernel, and measures how noisy timings are on this machine.

In [ ]:
# Setup: mount Drive, make sure the pinned torch/triton are installed, import archtune.
import importlib.metadata as md, os, subprocess, sys
PROJECT = "/content/drive/MyDrive/archtune"
NEED_GPU_STACK = True

if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)

def _ver(pkg):
    try:
        return md.version(pkg).split("+")[0]
    except md.PackageNotFoundError:
        return None

if NEED_GPU_STACK and (_ver("torch"), _ver("triton")) != ("2.6.0", "3.2.0"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.6.0", "triton==3.2.0"], check=True)
    if "torch" in sys.modules:
        raise SystemExit("torch was already imported: restart the session and run this cell again")

from archtune import config, gpu, model, compiler, run
ENV = gpu.capture()
MINI = config.mini_mode()
RESULTS = os.environ.get("ARCHTUNE_RESULTS") or os.path.join(PROJECT, "results_mini" if MINI else "results")
print(ENV["gpu_name"], "| torch", ENV["torch_version"], "| triton", ENV["triton_version"], "|", RESULTS)
if NEED_GPU_STACK and not gpu.pinned_ok(ENV):
    raise SystemExit("wrong torch/triton loaded: restart the session and run this cell again")

## Device

In [ ]:
import json, os
GPU_DIR = os.path.join(RESULTS, ENV["gpu_tag"]); os.makedirs(GPU_DIR, exist_ok=True)
live = gpu.live()
print(live)
ref_key = gpu.reference_for_name(live.name)
print("differences from the reference table:",
      gpu.verify(gpu.reference(ref_key), live) if ref_key else "no reference entry (peaks estimated from clocks)")
gpu.pip_freeze(os.path.join(GPU_DIR, "pip_freeze.txt"))
json.dump(gpu.device_properties(), open(os.path.join(GPU_DIR, "device_properties.json"), "w"), indent=2, default=str)

## Triton / PyTorch APIs on this stack

In [ ]:
API = gpu.api_report()
json.dump(API, open(os.path.join(GPU_DIR, "api_check.json"), "w"), indent=2, default=str)
for k, v in API.items():
    print(f"{k:34s} {v}")

## One compiled kernel

In [ ]:
mm = gpu.MATMUL
gpu.fresh_cache_dir()
cfg = {"BLOCK_SIZE_M": 64, "BLOCK_SIZE_N": 64, "BLOCK_SIZE_K": 32, "GROUP_SIZE_M": 8, "num_warps": 4, "num_stages": 3}
a, b, c = mm.make_inputs(1024, 1024, 1024)
res = gpu.compile_kernel(mm, (a, b, c), cfg)
print(res.status, f"compile {res.compile_time_s:.2f}s, cache_miss={res.cache_miss}")
print("n_regs", res.n_regs, "n_spills", res.n_spills, "shared_bytes", res.shared_bytes)
for name, desc in gpu.describe_compiled(res.compiled).items():
    print(f"  {name:22s} {desc}")

In [ ]:
import torch
mm.launcher(a, b, c, cfg)(); torch.cuda.synchronize()
ok, err = gpu.check_correct(c, mm.reference(a, b))
print("correct:", ok, "max_abs_err:", err)
print(res.asm["ptx"][:1500])

## SASS

In [ ]:
for tool in ("cuobjdump", "nvdisasm", "ptxas"):
    print(tool, "->", gpu.find_cuda_tool(tool))
sass = gpu.disassemble(res.asm.get("cubin"))
print(sass[:2000] if sass else "no disassembler found; PTX only")

## Timing noise

One config timed 20 times. Differences smaller than twice this coefficient of variation are treated
as ties everywhere else in the project.

In [ ]:
a, b, c = mm.make_inputs(4096, 4096, 4096)
big = {**cfg, "BLOCK_SIZE_M": 128, "BLOCK_SIZE_N": 128}
noise = gpu.noise_check(mm.launcher(a, b, c, big), n=5 if MINI else config.BENCH.noise_runs)
noise.update(shape="4096x4096x4096", cfg=big, gpu_tag=ENV["gpu_tag"], stack_id=ENV["stack_id"])
json.dump(noise, open(os.path.join(GPU_DIR, "noise.json"), "w"), indent=2)
print(f"CV = {noise['cv']:.4f}; differences under {noise['tie_threshold']:.2%} are ties")